# AI-Based Business Location Selector: K-Means Clustering & Suitability Scoring
**Objective:** Segment candidate locations using unsupervised K-Means and apply the multi-criteria suitability formula to rank locations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

data_path = Path("../data/business_location_synthetic_dataset.csv")
if not data_path.exists():
    data_path = Path("business_location_synthetic_dataset.csv")
df = pd.read_csv(data_path)

cluster_cols = ["monthly_rent", "daily_footfall", "competitor_count", "accessibility_score", "population"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_cols])

## 1. Elbow Method & Silhouette Analysis for Optimal k

In [ ]:
k_range = range(2, 8)
inertias = []
silhs = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhs.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(k_range, inertias, marker='o', color='b')
ax[0].set_title('Elbow Method (Inertia vs. k)')
ax[0].set_xlabel('k')
ax[0].set_ylabel('Inertia')

ax[1].bar(k_range, silhs, color='teal')
ax[1].set_title('Silhouette Score vs. k')
ax[1].set_xlabel('k')
ax[1].set_ylabel('Silhouette Score')
plt.show()

## 2. Multi-Criteria Suitability Scoring Demo (Cafe, Budget = Rs. 35,000)

In [ ]:
# Filter Cafe
df_cafe = df[df["business_type"] == "Cafe"].copy()

# Weights: S = 0.30D + 0.20A + 0.15F + 0.15P + 0.10C + 0.10R
def compute_score(row, budget=35000):
    D = row["demand_score"]
    A = ((row["accessibility_score"] - 1) / 9) * 100
    F = row["footfall_score"]
    P = row["mobility_score"]
    C = row["competition_score"]
    
    # Rent affordability
    rent = row["monthly_rent"]
    if rent <= 0.5 * budget:
        R = 100.0
    elif rent <= budget:
        R = 100.0 - 60.0 * ((rent - 0.5 * budget) / (0.5 * budget))
    else:
        R = max(0.0, 40.0 - 80.0 * ((rent - budget) / budget))
        
    S = 0.30*D + 0.20*A + 0.15*F + 0.15*P + 0.10*C + 0.10*R
    return S

df_cafe["Suitability_Score"] = df_cafe.apply(compute_score, axis=1)
df_ranked = df_cafe.sort_values(by="Suitability_Score", ascending=False).reset_index(drop=True)
df_ranked[["location_name", "monthly_rent", "daily_footfall", "target_demand", "Suitability_Score"]].head(10)